# A01 · LLM APIs, Structured Output & Cost

**Track:** Advanced (lesson 1 of 10) · **Time:** ~2.5 h · **Mode:** runs OFFLINE for free, or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** Intermediate track (tokens, transformers), basic Python/Pydantic · **You will build:** a validated invoice-extraction pipeline with a repair loop, retries, and a cost calculator

---
## 🎯 Why this matters
An LLM call is the least reliable function in your codebase: it can time out, rate-limit, return prose where you wanted JSON, return JSON that
parses but is *wrong*, and cost 20x more than you budgeted. Production LLM engineering is mostly the discipline of wrapping that
function so the rest of your system can treat it as a typed, bounded-cost, retryable call.

**Real scenario.** *Kauri Finance* (an accounts-payable team in Auckland) receives ~50,000 supplier invoices a month as email text and PDFs-turned-text.
Today clerks key them into the ERP. You are asked to extract `vendor, invoice_number, issue_date, line items, GST, total` into the ERP's schema.
Success metric: **>= 99% of invoices land in the ERP with a valid, internally consistent record; the rest go to a human queue; cost per invoice is known in advance.**

## 🧭 Learning objectives
By the end you can:
1. Describe the anatomy of a Messages API request/response (roles, content blocks, `stop_reason`, `usage`) and where tokens are billed.
2. Estimate and project LLM cost from token counts using a price table, and say which levers (model, output length, retries) matter most.
3. Extract structured data with a Pydantic schema, a tolerant JSON parser, and a **validate-then-repair loop**; measure first-pass vs final validity.
4. Implement retries with exponential backoff + jitter, and decide which errors are retryable.
5. Explain why a schema-valid answer can still be wrong, and add cross-field and grounding checks.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept: the Messages API in one picture

```
 request  = model + max_tokens + [system] + messages[ {role, content} ... ] (+ tools, tool_choice, temperature...)
 response = content[ blocks ] + stop_reason + usage{input_tokens, output_tokens}
```

| Piece | What to remember |
|---|---|
| `system` | Instructions that frame the whole conversation (role, rules, output format). Separate from `messages`. |
| `messages` | Alternating `user` / `assistant` turns. The API is **stateless**: you re-send the history every call, and you pay for it every call. |
| `content` blocks | `text`, `tool_use`, (and thinking blocks for reasoning). A reply is a *list* of blocks, not a string. |
| `stop_reason` | `end_turn` (finished), `max_tokens` (**truncated** - output may be invalid JSON!), `tool_use`, `refusal`. |
| `usage` | `input_tokens` and `output_tokens`: this is what you are billed for. Output tokens cost several times more than input tokens. |

Reference (needs a key, so shown, not run - check the current docs for parameters):

```python
import anthropic
client = anthropic.Anthropic()                      # reads ANTHROPIC_API_KEY; the SDK also retries some errors for you
resp = client.messages.create(
    model="<model id>", max_tokens=1024,
    system="You extract invoices into JSON.",
    messages=[{"role": "user", "content": invoice_text}],
)
resp.content[0].text, resp.stop_reason, resp.usage.input_tokens, resp.usage.output_tokens
```

**Three ways to get structure out of a model**, from weakest to strongest guarantee:
1. *Ask nicely* ("reply with JSON") + parse + validate + repair. Works anywhere; you own the safety net. **We build this.**
2. *Forced tool use*: define a tool whose `input_schema` is your schema and set `tool_choice` to that tool; the model's "arguments" are your JSON.
3. *Provider-native structured outputs* (constrained decoding) where available. Check current docs for what is supported.

Even with 2 and 3, you still validate: constrained decoding guarantees **shape**, not **truth**.

## 2 · Build it, part 1: the world, the schema, the cost model
First the data. We generate a fixed-seed batch of synthetic supplier invoices (free text, like an email body). The generator also keeps the
ground truth so we can *measure* extraction accuracy instead of eyeballing it.

In [ ]:
import json, re, random, math, datetime as dt
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator
from agentkit import get_llm, say, Reply, PolicyLLM
from agentkit.llm import DEFAULT_MODEL
from agentkit.trace import PRICES

rng = random.Random(7)
VENDORS = ["Kauri Office Supplies Ltd", "Pohutukawa Catering Co", "Southern Alps Freight Ltd", "Harbour City IT Services",
           "Tui Cleaning Services", "Manuka Print & Signage"]
CATALOGUE = [("A4 copy paper (box)", 8.50), ("Courier - Auckland metro", 14.90), ("Monthly IT support retainer", 450.00),
             ("Catering - morning tea (per head)", 12.75), ("Toner cartridge", 96.20), ("Office cleaning (hour)", 38.00),
             ("Business cards (500)", 61.40), ("Freight - pallet", 120.00)]
MONTHS = ["January","February","March","April","May","June","July","August","September","October","November","December"]

def make_invoice(n: int) -> dict:
    items = [{"description": d, "quantity": rng.randint(1, 12), "unit_price": p} for d, p in rng.sample(CATALOGUE, rng.randint(2, 4))]
    sub = round(sum(i["quantity"] * i["unit_price"] for i in items), 2)
    gst = round(sub * 0.15, 2)
    date = dt.date(2025, rng.randint(1, 12), rng.randint(1, 28))
    truth = {"invoice_number": f"INV-{2000 + n}", "vendor": rng.choice(VENDORS), "issue_date": date.isoformat(), "currency": "NZD",
             "line_items": items, "gst": gst, "total": round(sub + gst, 2)}
    lines = "\n".join(f"  {i['quantity']} x {i['description']} @ {i['unit_price']:.2f}" for i in items)
    text = (f"INVOICE {truth['invoice_number']}\nFrom: {truth['vendor']}\nDate: {date.day} {MONTHS[date.month-1]} {date.year}\n"
            f"Items:\n{lines}\nSubtotal: {sub:.2f}\nGST (15%): {gst:.2f}\nTotal due: NZD {truth['total']:.2f}\nPayment terms: 20th of next month")
    return {"id": truth["invoice_number"], "text": text, "truth": truth}

INVOICES = [make_invoice(n) for n in range(1, 13)]
INV_BY_ID = {i["id"]: i for i in INVOICES}
print(INVOICES[0]["text"])

### The target schema
The schema is the contract with the ERP. Two design choices worth copying:
* `extra="forbid"`: unknown fields are an error, not silently dropped.
* A **cross-field validator**: `sum(quantity x unit_price) + gst == total`. Types can't express this; a validator can, and it catches a whole class of
  silent extraction errors (misread digits, dropped lines).

In [ ]:
class LineItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    description: str = Field(min_length=1)
    quantity: int = Field(gt=0)
    unit_price: float = Field(ge=0)

class Invoice(BaseModel):
    model_config = ConfigDict(extra="forbid")
    invoice_number: str = Field(pattern=r"^INV-\d{4}$")
    vendor: str = Field(min_length=2)
    issue_date: dt.date                      # ISO 8601 only: "2025-03-14"
    currency: Literal["NZD", "AUD", "USD"]
    line_items: list[LineItem] = Field(min_length=1)
    gst: float = Field(ge=0)
    total: float = Field(gt=0)

    @model_validator(mode="after")
    def totals_add_up(self):
        subtotal = sum(i.quantity * i.unit_price for i in self.line_items)
        if abs(subtotal + self.gst - self.total) > 0.01:
            raise ValueError(f"line items ({subtotal:.2f}) + gst ({self.gst:.2f}) != total ({self.total:.2f})")
        return self

schema = Invoice.model_json_schema()
print("schema properties:", list(schema["properties"]))
# sanity: the ground truth we generated is itself valid under the contract
assert all(Invoice.model_validate(i["truth"]) for i in INVOICES)

### A cost model before you write any more code
Cost = `input_tokens x price_in + output_tokens x price_out`. Real tokenizers differ by model; for *planning* a rough rule of thumb of
~4 characters per English token is good enough (use the API's token-counting endpoint or the `usage` field for exact numbers). The price table is
`agentkit.trace.PRICES` ($ per million tokens: input, output). Never hard-code prices in application code; keep one table and read it.

In [ ]:
def est_tokens(text: str) -> int:
    return max(1, math.ceil(len(text) / 4))          # planning heuristic, not a tokenizer

def cost_usd(model: str, tokens_in: int, tokens_out: int) -> float:
    p_in, p_out = PRICES[model]
    return (tokens_in * p_in + tokens_out * p_out) / 1e6

SYSTEM = ("You extract supplier invoices into JSON for an ERP. Reply with ONE JSON object and nothing else. "
          "Dates are ISO 8601 (YYYY-MM-DD). Amounts are plain numbers (no currency symbols). "
          f"JSON Schema:\n{json.dumps(schema)}")

sample_in = est_tokens(SYSTEM) + est_tokens(INVOICES[0]["text"])
sample_out = est_tokens(json.dumps(INVOICES[0]["truth"]))
print(f"one invoice ~ {sample_in} input tokens (system prompt is {est_tokens(SYSTEM)} of them!), ~{sample_out} output tokens")
for m, (pi, po) in PRICES.items():
    print(f"{m:<18} ${cost_usd(m, sample_in, sample_out):.5f} / invoice   -> {50_000 * cost_usd(m, sample_in, sample_out):>8.2f} per 50k invoices")

assert cost_usd("claude-opus-5-5", 1000, 1000) > cost_usd("claude-haiku-5-5", 1000, 1000)
assert cost_usd(DEFAULT_MODEL, 0, 1_000_000) == PRICES[DEFAULT_MODEL][1]

Two things to notice. (1) The **system prompt + schema is re-sent on every call** and dominates the input; that is exactly what prompt caching (A09)
targets. (2) Output tokens are the expensive direction, so *asking for less output* (compact JSON, no explanations) is a direct cost lever.

## 3 · Build it, part 2: the extraction pipeline with a repair loop
The recipe:
```
 call model ─▶ tolerant parse (strip fences / prose) ─▶ Pydantic validate ─▶ OK
                                │ fail                          │ fail
                                └──────────────┬────────────────┘
                   append the bad answer + the *specific* errors, ask for a fix (max N times)
                                └─▶ still failing ⇒ dead-letter queue for a human
```
Why feed back the *specific* error? The model is good at fixing "issue_date: Input should be a valid date, got 14/03/2025" and bad at
guessing what you disliked. Why a cap? Each repair is another paid call, and some inputs will never succeed.

**The model offline.** We need a stand-in that behaves like a real model's *failure modes*. `fake_model` below extracts correctly but, on the first
attempt for particular invoices, makes the mistakes real models make (markdown fences, chatty preamble, `14/03/2025` dates, `"NZD 129.50"` amounts, an
arithmetic slip, an invented field). On a repair turn it fixes the problem, except for one invoice that never gets fixed. LIVE mode swaps in Claude.

In [ ]:
DEFECTS = {"INV-2002": "fence", "INV-2003": "prose", "INV-2004": "bad_date", "INV-2006": "amount_text",
           "INV-2008": "wrong_sum", "INV-2009": "extra_field", "INV-2011": "stubborn"}

def fake_model(messages, system, tools):
    inv_id = re.search(r"INV-\d{4}", messages[0]["content"]).group(0)
    truth = json.loads(json.dumps(INV_BY_ID[inv_id]["truth"]))
    attempt = sum(1 for m in messages if m["role"] == "assistant")
    defect = DEFECTS.get(inv_id)
    if defect == "stubborn" or (defect and attempt == 0):
        if defect in ("bad_date", "stubborn"):
            y, mth, d = truth["issue_date"].split("-"); truth["issue_date"] = f"{d}/{mth}/{y}"
        if defect == "amount_text":  truth["total"] = f"NZD {truth['total']:.2f}"
        if defect == "wrong_sum":    truth["total"] = round(truth["total"] + 10, 2)
        if defect == "extra_field":  truth["notes"] = "paid by direct credit"
    body = json.dumps(truth)
    if defect == "fence" and attempt == 0:  body = f"```json\n{body}\n```"
    if defect == "prose" and attempt == 0:  body = f"Sure! Here is the extracted invoice:\n{body}\nLet me know if you need anything else."
    tin = est_tokens(system) + sum(est_tokens(m["content"]) for m in messages if isinstance(m["content"], str))
    return Reply([{"type": "text", "text": body}], "end_turn", {"input_tokens": tin, "output_tokens": est_tokens(body)})

llm = get_llm(policy=fake_model)

In [ ]:
def parse_json(raw: str) -> dict:
    """Tolerant parser: take the outermost {...}; this absorbs ```json fences and chatty preambles."""
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("no JSON object found in the reply")
    return json.loads(raw[start:end + 1])

def format_errors(e: Exception) -> str:
    if isinstance(e, ValidationError):
        return "\n".join(f"- {'.'.join(str(p) for p in x['loc']) or 'record'}: {x['msg']}" for x in e.errors())
    return f"- {type(e).__name__}: {e}"

def extract(llm, invoice_text: str, max_repairs: int = 2) -> dict:
    messages = [{"role": "user", "content": f"Extract this invoice:\n\n{invoice_text}"}]
    tin = tout = 0
    errors = []
    for attempt in range(1, max_repairs + 2):
        reply = llm.chat(messages, system=SYSTEM, max_tokens=800)
        tin += reply.usage["input_tokens"]; tout += reply.usage["output_tokens"]
        if reply.stop_reason == "max_tokens":                 # truncated JSON: repairing the *text* will not help
            errors.append("truncated"); break
        try:
            invoice = Invoice.model_validate(parse_json(reply.text))
            return {"ok": True, "invoice": invoice, "attempts": attempt, "errors": errors, "tin": tin, "tout": tout}
        except ValueError as e:                                # JSONDecodeError and pydantic ValidationError are both ValueErrors
            feedback = format_errors(e); errors.append(feedback)
            messages += [{"role": "assistant", "content": reply.text},
                         {"role": "user", "content": f"Your JSON failed validation:\n{feedback}\nReturn the corrected JSON object only."}]
    return {"ok": False, "invoice": None, "attempts": attempt, "errors": errors, "tin": tin, "tout": tout}

r = extract(llm, INV_BY_ID["INV-2004"]["text"])
print("attempts:", r["attempts"], "| error fed back to the model:\n" + r["errors"][0])
assert r["ok"] and r["attempts"] == 2 and r["invoice"].issue_date == dt.date.fromisoformat(INV_BY_ID["INV-2004"]["truth"]["issue_date"])

## 4 · Use it: run the whole batch and *measure*
Three numbers define a structured-output pipeline: **first-pass validity** (how often the prompt alone works), **final validity** (after repair), and
**accuracy against truth** (are the valid ones *right*?). Add cost per invoice and the dead-letter count and you can make a go/no-go decision.

In [ ]:
results = {i["id"]: extract(llm, i["text"]) for i in INVOICES}
ok = [k for k, v in results.items() if v["ok"]]
first_pass = [k for k, v in results.items() if v["ok"] and v["attempts"] == 1]
dead = [k for k, v in results.items() if not v["ok"]]

def matches_truth(inv_id: str) -> bool:
    got = results[inv_id]["invoice"].model_dump(mode="json")
    return got == INV_BY_ID[inv_id]["truth"]

correct = [k for k in ok if matches_truth(k)]
tin = sum(v["tin"] for v in results.values()); tout = sum(v["tout"] for v in results.values())
print(f"first-pass valid : {len(first_pass)}/{len(INVOICES)}")
print(f"final valid      : {len(ok)}/{len(INVOICES)}   dead-letter -> human queue: {dead}")
print(f"accurate (== truth) among valid: {len(correct)}/{len(ok)}")
print(f"avg attempts     : {sum(v['attempts'] for v in results.values()) / len(results):.2f}")
for m in (DEFAULT_MODEL, "claude-haiku-5-5"):
    print(f"cost on {m:<17}: ${cost_usd(m, tin, tout) / len(INVOICES):.5f}/invoice (retries included)")

assert len(ok) == 11 and dead == ["INV-2011"]
assert len(correct) == len(ok), "valid-but-wrong answers slipped through"
assert len(first_pass) == 7 and len(first_pass) < len(ok),"the repair loop should rescue some invoices"

**Reading the result.** The repair loop lifted validity from 7/12 to 11/12 (fences and chatty preambles never needed a repair: the tolerant parser absorbed them) for a small extra token bill (look at `avg attempts`), and the stubborn invoice went to
the human queue instead of corrupting the ERP. In LIVE mode your first-pass rate will differ - that is the point of measuring it. If first-pass is low,
fix the *prompt/schema* (A02); do not hide it behind more retries.

## 5 · Reliability: timeouts, retries, backoff
Network and capacity errors are normal at 50k calls a month. The rule: **retry what is transient, fail fast on what is your bug.**

| Error class | Examples | Retry? |
|---|---|---|
| Transient | connection reset, timeout, HTTP 429 (rate limit), 5xx / overloaded | **Yes**, with exponential backoff + jitter (and honour any `retry-after` header) |
| Client bug | HTTP 400 invalid request, 401/403 auth, 404 bad model name | **No**: retrying cannot fix it |
| Output problem | invalid JSON, schema violation | Not a *transport* retry: use the repair loop above |

**Jitter** matters: if 1,000 workers all retry at exactly 1 s, 2 s, 4 s they stampede the API in synchronised waves. "Full jitter" picks a random delay in `[0, min(cap, base * 2**n)]`.
The Anthropic SDK retries some errors automatically and lets you configure `max_retries` and `timeout` on the client; write your own wrapper when you need to add logging, budgets or a fallback model.

In [ ]:
class TransientError(Exception): ...
class ClientError(Exception): ...

def with_retries(fn, retries=4, base=0.5, cap=8.0, sleep=lambda s: None, rand=random.Random(0)):
    """Call fn(); retry TransientError with full-jitter exponential backoff. `sleep` is injected so tests don't wait."""
    delays = []
    for n in range(retries + 1):
        try:
            return fn(), delays
        except TransientError:
            if n == retries:
                raise
            d = rand.uniform(0, min(cap, base * 2 ** n)); delays.append(round(d, 3)); sleep(d)

state = {"calls": 0}
def flaky_call():
    state["calls"] += 1
    if state["calls"] <= 3:
        raise TransientError("HTTP 529 overloaded")
    return llm.chat([{"role": "user", "content": INVOICES[0]["text"]}], system=SYSTEM)

reply, delays = with_retries(flaky_call)
print("succeeded after", state["calls"], "calls; backoff delays (s):", delays)
assert state["calls"] == 4 and len(delays) == 3 and all(d <= 8.0 for d in delays)

def bad_request():
    raise ClientError("HTTP 400: max_tokens too large")
try:
    with_retries(bad_request)
except ClientError as e:
    print("client errors are NOT retried:", e)

### Streaming (concept)
For a human waiting on an answer, **time-to-first-token (TTFT)** matters more than total time. With streaming the API sends tokens as they are
generated, so the UI can paint text after ~1 s instead of after the whole answer. For *machine* consumers (our ERP pipeline) there is nothing to
show a user, so we do not stream; we want the complete validated object. Reference (needs a key):

```python
with client.messages.stream(model="<model id>", max_tokens=1024, messages=[...]) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    final = stream.get_final_message()          # contains usage + stop_reason too
```
Streaming and structured output interact: you can only validate once the object is complete, so stream *to the user*, validate *before acting*.

## 6 · Break it
Three ways structured output goes wrong *silently*. Each has a guard.

### 6a · Valid shape, wrong facts
The schema accepts `vendor="Acme"` even if "Acme" appears nowhere in the invoice. Constrained decoding would also happily produce this. Guard: **grounding checks** against the source.

In [ ]:
def grounding_issues(inv: Invoice, source: str) -> list[str]:
    issues = []
    if inv.vendor.lower() not in source.lower():
        issues.append(f"vendor '{inv.vendor}' not found in source")
    if inv.invoice_number not in source:
        issues.append(f"invoice_number {inv.invoice_number} not in source")
    for li in inv.line_items:
        if li.description.lower() not in source.lower():
            issues.append(f"line item '{li.description}' not in source")
    if f"{inv.total:.2f}" not in source:
        issues.append(f"total {inv.total:.2f} not in source")
    return issues

good = results["INV-2001"]["invoice"]
hallucinated = good.model_copy(update={"vendor": "Acme Holdings Ltd"})       # still schema-valid!
assert Invoice.model_validate(hallucinated.model_dump()) and grounding_issues(good, INV_BY_ID["INV-2001"]["text"]) == []
print("hallucinated vendor ->", grounding_issues(hallucinated, INV_BY_ID["INV-2001"]["text"]))
assert grounding_issues(hallucinated, INV_BY_ID["INV-2001"]["text"]) != []
bad_extractions = [k for k in ok if grounding_issues(results[k]["invoice"], INV_BY_ID[k]["text"])]
assert bad_extractions == [], bad_extractions

### 6b · Truncation (`stop_reason == "max_tokens"`)
If `max_tokens` is too small the JSON is cut mid-object. A repair prompt cannot fix a response that ran out of budget; it will just be truncated again at a
higher cost. Treat `max_tokens` as its own failure class: raise the cap, shrink the output (fewer fields), or split the document.

In [ ]:
def truncating_model(messages, system, tools):
    full = json.dumps(INV_BY_ID["INV-2001"]["truth"])
    return Reply([{"type": "text", "text": full[:60]}], "max_tokens", {"input_tokens": 500, "output_tokens": 20})

r = extract(PolicyLLM(truncating_model), INV_BY_ID["INV-2001"]["text"])
print(r["ok"], r["attempts"], r["errors"])
assert not r["ok"] and r["attempts"] == 1, "must not burn repair calls on truncation"

### 6c · Retry storms and unbounded cost
Worst case cost per invoice = `(1 + max_repairs)` calls, and the *history grows* each repair (the bad answer is re-sent). A batch job with no cap on
repairs, no per-job budget and no dead-letter queue turns one bad template into a bill. Make the guard explicit:

In [ ]:
def worst_case_cost(model, tin_per_call, tout_per_call, max_repairs):
    # each repair re-sends the previous answer + feedback: input grows roughly by tout + ~60 tokens per round
    total_in = sum(tin_per_call + k * (tout_per_call + 60) for k in range(max_repairs + 1))
    return cost_usd(model, total_in, tout_per_call * (max_repairs + 1))

base_cost = cost_usd(DEFAULT_MODEL, sample_in, sample_out)
wc = worst_case_cost(DEFAULT_MODEL, sample_in, sample_out, 2)
print(f"happy path ${base_cost:.5f}  vs worst case with 2 repairs ${wc:.5f}  ({wc / base_cost:.1f}x)")
assert 2.9 < wc / base_cost < 4.5

## 7 · Same idea via forced tool use (reference)
Instead of "reply with JSON", declare the schema as a tool and force the model to call it. The arguments arrive as an already-parsed dict in a `tool_use` block, which removes
the fence/prose failure class, but you still run Pydantic validation (types of values, your cross-field rules, grounding).

```python
resp = client.messages.create(
    model="<model id>", max_tokens=1024,
    tools=[{"name": "record_invoice", "description": "Record the extracted invoice.", "input_schema": Invoice.model_json_schema()}],
    tool_choice={"type": "tool", "name": "record_invoice"},
    messages=[{"role": "user", "content": f"Extract this invoice:\n\n{text}"}],
)
block = next(b for b in resp.content if b.type == "tool_use")
invoice = Invoice.model_validate(block.input)
```
*Reference: check the current docs for `tool_choice` options and any native structured-output features.* In this repo's `agentkit`, the same shape is
produced offline by `call("record_invoice", **fields)`; the validation code is identical.

## 8 · Exercises
**Practice (30 min)**
1. Add an optional `due_date: dt.date | None` field. Extend `make_invoice`'s text with "Due: ..." and make `fake_model` emit it. Re-run the batch; do validity and accuracy hold?
2. Add a `purchase_order: str | None` field with a pattern `PO-\d{5}`. What should the pipeline do when the source has *no* PO: omit, `null`, or invent? Show with a test which your prompt+schema produce.

**Stretch (60 min)**
3. Implement a **batch cost guard**: stop the batch and raise if cumulative estimated cost exceeds a budget; assert it triggers when you set the budget to the cost of 5 invoices.
4. Add a **fallback model** to `extract`: after one failed repair, retry once on a stronger model (`PRICES` key). Compare the cost with always using the strong model.

**Challenge (2 h, needs a key)**
5. Run the same 12 invoices LIVE with (a) prompt-only JSON and (b) forced tool use. Report first-pass validity, final validity, and cost per invoice for each. Which failure classes disappear in (b)? Which remain?

In [ ]:
# Starter for exercise 3 - fill in and make the assertions pass
class BudgetExceeded(Exception): ...

def run_batch_with_budget(llm, invoices, budget_usd, model=DEFAULT_MODEL):
    """Process invoices in order; raise BudgetExceeded before starting an invoice if spend so far >= budget."""
    raise NotImplementedError

# spent_for_5 = sum(cost_usd(DEFAULT_MODEL, results[i["id"]]["tin"], results[i["id"]]["tout"]) for i in INVOICES[:5])
# try:
#     run_batch_with_budget(llm, INVOICES, budget_usd=spent_for_5 + 1e-9)
#     raise AssertionError("budget guard did not trigger")
# except BudgetExceeded:
#     pass

## ✅ Checkpoint (answer before moving on)
1. The API is stateless. What does that imply for the cost of a 10-turn conversation, and for each repair attempt in our loop?
2. Why do we parse with a *tolerant* extractor and validate with a *strict* schema, rather than making both strict?
3. Our batch shows 11/12 valid and 11/11 accurate. Name two ways "valid" could still be wrong in production that the checks in this lesson would **not** catch.
4. Which errors do you retry with backoff, which do you send through the repair loop, and which do you dead-letter?

<details><summary>Answers</summary>

1. Every call re-sends the whole history, so input tokens grow roughly quadratically in total over a conversation; each repair re-sends the system prompt, the document, the bad answer and the feedback, so repair calls cost *more* than the first call.
2. Tolerance on input form (fences, preamble) costs nothing and avoids pointless repair calls; strictness on *content* is where errors are expensive. Be liberal in what you accept syntactically, strict about what you let into the ERP.
3. Examples: a value that exists in the source but belongs to a different field (e.g. the shipping total used as the invoice total, with consistent line items); a duplicated invoice; a wrong currency that is in the allowed set; a swapped vendor/customer. These need business rules, deduplication, or a human/second-model review (A04).
4. Transient transport errors: retry with backoff + jitter. Output-format/validation errors: repair loop with specific feedback, capped. Client bugs (400/401): fail fast and alert. `max_tokens` truncation and inputs that still fail after N repairs: dead-letter for a human (and log them, they are your next eval cases).
</details>

## 📌 Key takeaways
- Treat an LLM call as an unreliable, metered function: wrap it with **validation, bounded retries, and a cost model**.
- Structure comes from *schema + tolerant parse + strict validate + targeted repair*; native structured outputs reduce failure classes but never replace validation.
- A schema guarantees shape, not truth: add cross-field rules and grounding checks, and route what fails to humans.
- Measure **first-pass validity, final validity, accuracy, attempts and cost**. Fix prompts for first-pass rate; do not hide problems behind retries.
- Output tokens and repeated system prompts are the main cost levers; prices live in one table (`PRICES`), never in application code.

## 📚 Further reading
- Anthropic docs: *Messages API*, *Tool use*, *Streaming messages*, *Errors*, and the pricing page (check current versions).
- Pydantic docs: *Validators* and *JSON Schema*.
- Marc Brooker, *Exponential Backoff And Jitter* (AWS Architecture Blog, 2015).

**Next → A02 · Prompting & Context Engineering** (what to put in the prompt, how to budget the context window, and how to A/B test prompts)